In [ ]:
%%sql

--Analisis de la evolucion temporal y variacion de potencia radiativa (FRP) en los clusters de incendios activos.

DROP TABLE IF EXISTS gold_fire_evolution;

--creacion de la tabla gold para el seguimiento de la evolucion de focos de fuego
CREATE TABLE gold_fire_evolution (
    fire_id STRING,
    cluster_id STRING,
    latitude DOUBLE,
    longitude DOUBLE,
    acq_date DATE,
    acq_time INT,
    fire_detection_timestamp TIMESTAMP,
    bright_ti4 DOUBLE,
    bright_ti5 DOUBLE,
    fire_radiative_power DOUBLE,
    previous_frp DOUBLE,
    frp_change_mw DOUBLE,
    evolution_trend STRING,
    confidence STRING,
    daynight STRING,
    detection_date DATE,
    detection_hour INT,
    detection_minute INT,
    detection_year INT,
    detection_month INT,
    detection_day INT,
    fire_intensity_level STRING,
    fire_intensity_score INT,
    cluster_first_detection_timestamp TIMESTAMP,
    cluster_last_detection_timestamp TIMESTAMP,
    cluster_detection_count BIGINT,
    cluster_centroid_latitude DOUBLE,
    cluster_centroid_longitude DOUBLE,
    cluster_max_frp DOUBLE,
    cluster_avg_frp DOUBLE,
    landing_source_file STRING,
    ingestion_timestamp TIMESTAMP,
    updated_source_file STRING,
    updated_timestamp TIMESTAMP,
    gold_updated_at TIMESTAMP
);

--Informacion de clusters

--informacion agregada del cluster para unirla a cada deteccion individual
CREATE OR REPLACE TEMP VIEW cluster_information AS

SELECT
    cluster_id,
    first_detection_timestamp
        AS cluster_first_detection_timestamp,
    last_detection_timestamp
        AS cluster_last_detection_timestamp,
    detection_count
        AS cluster_detection_count,
    centroid_latitude
        AS cluster_centroid_latitude,
    centroid_longitude
        AS cluster_centroid_longitude,
    max_fire_radiative_power
        AS cluster_max_frp,
    avg_fire_radiative_power
        AS cluster_avg_frp
FROM silver_fire_clusters;


--Detecciones

--vista para unificar datos de la nasa calculando niveles de intensidad y claves unicas
CREATE OR REPLACE TEMP VIEW fire_evolution_base AS

SELECT
    SHA2(
        CONCAT_WS(
            '|',
            CAST(f.latitude AS STRING),
            CAST(f.longitude AS STRING),
            CAST(f.fire_detection_timestamp AS STRING)
        ),
        256
    ) AS fire_id,
    f.cluster_id,
    CAST(f.latitude AS DOUBLE) AS latitude,
    CAST(f.longitude AS DOUBLE) AS longitude,
    f.acq_date,
    f.acq_time,
    f.fire_detection_timestamp,
    CAST(f.bright_ti4 AS DOUBLE) AS bright_ti4,
    CAST(f.bright_ti5 AS DOUBLE) AS bright_ti5,
    CAST(f.fire_radiative_power AS DOUBLE)
        AS fire_radiative_power,
    f.confidence,
    f.daynight,
    DATE(f.fire_detection_timestamp)
        AS detection_date,
    HOUR(f.fire_detection_timestamp)
        AS detection_hour,
    MINUTE(f.fire_detection_timestamp)
        AS detection_minute,
    YEAR(f.fire_detection_timestamp)
        AS detection_year,
    MONTH(f.fire_detection_timestamp)
        AS detection_month,
    DAY(f.fire_detection_timestamp)
        AS detection_day,
    CASE
        WHEN f.fire_radiative_power >= 100
            THEN 'VERY_HIGH'
        WHEN f.fire_radiative_power >= 50
            THEN 'HIGH'
        WHEN f.fire_radiative_power >= 10
            THEN 'MEDIUM'
        WHEN f.fire_radiative_power >= 0
            THEN 'LOW'
        ELSE 'UNKNOWN'
    END AS fire_intensity_level,
    CASE
        WHEN f.fire_radiative_power >= 100
            THEN 4
        WHEN f.fire_radiative_power >= 50
            THEN 3
        WHEN f.fire_radiative_power >= 10
            THEN 2
        WHEN f.fire_radiative_power >= 0
            THEN 1
        ELSE 0
    END AS fire_intensity_score,
    c.cluster_first_detection_timestamp,
    c.cluster_last_detection_timestamp,
    c.cluster_detection_count,
    c.cluster_centroid_latitude,
    c.cluster_centroid_longitude,
    c.cluster_max_frp,
    c.cluster_avg_frp,
    f.landing_source_file,
    f.ingestion_timestamp

FROM silver_nasa_fires f

LEFT JOIN cluster_information c
    ON f.cluster_id = c.cluster_id

WHERE f.fire_detection_timestamp IS NOT NULL
  AND f.latitude IS NOT NULL
  AND f.longitude IS NOT NULL;


--Evolucion

--calculo la diferencia de potencia radiativa frente al registro previo del mismo cluster
CREATE OR REPLACE TEMP VIEW fire_evolution_source AS

WITH lagged_data AS (
    SELECT
        *,
        LAG(fire_radiative_power) OVER (
            PARTITION BY cluster_id
            ORDER BY fire_detection_timestamp ASC
        ) AS previous_frp
    FROM fire_evolution_base
)

SELECT
    fire_id,
    cluster_id,
    latitude,
    longitude,
    acq_date,
    acq_time,
    fire_detection_timestamp,
    bright_ti4,
    bright_ti5,
    fire_radiative_power,
    previous_frp,
    ROUND(
        COALESCE(
            fire_radiative_power - previous_frp,
            0
        ),
        2
    ) AS frp_change_mw,
    CASE
        WHEN previous_frp IS NULL
            THEN 'NEW'
        WHEN fire_radiative_power > previous_frp
            THEN 'WORSENING'
        WHEN fire_radiative_power < previous_frp
            THEN 'IMPROVING'
        ELSE 'STABLE'
    END AS evolution_trend,
    confidence,
    daynight,
    detection_date,
    detection_hour,
    detection_minute,
    detection_year,
    detection_month,
    detection_day,
    fire_intensity_level,
    fire_intensity_score,
    cluster_first_detection_timestamp,
    cluster_last_detection_timestamp,
    cluster_detection_count,
    cluster_centroid_latitude,
    cluster_centroid_longitude,
    cluster_max_frp,
    cluster_avg_frp,
    landing_source_file,
    ingestion_timestamp,
    CAST(NULL AS STRING)
        AS updated_source_file,
    CAST(NULL AS TIMESTAMP)
        AS updated_timestamp,
    current_timestamp()
        AS gold_updated_at

FROM lagged_data;


--Carga

--insercion final de los registros procesados en la tabla gold de evolucion
INSERT INTO gold_fire_evolution

SELECT

    fire_id,
    cluster_id,
    latitude,
    longitude,
    acq_date,
    acq_time,
    fire_detection_timestamp,
    bright_ti4,
    bright_ti5,
    fire_radiative_power,
    previous_frp,
    frp_change_mw,
    evolution_trend,
    confidence,
    daynight,
    detection_date,
    detection_hour,
    detection_minute,
    detection_year,
    detection_month,
    detection_day,
    fire_intensity_level,
    fire_intensity_score,
    cluster_first_detection_timestamp,
    cluster_last_detection_timestamp,
    cluster_detection_count,
    cluster_centroid_latitude,
    cluster_centroid_longitude,
    cluster_max_frp,
    cluster_avg_frp,
    landing_source_file,
    ingestion_timestamp,
    updated_source_file,
    updated_timestamp,
    gold_updated_at

FROM fire_evolution_source;